# Домашнее задание. Применение статистических критериев

В этом задании вы потренируетесь подбирать и применять статистические критерии под конкретную задачу: доверительные интервалы, одновыборочные и двухвыборочные тесты, дисперсионный анализ и анализ долей.

Инструкции:
- Скопируйте ноутбук себе и заполните ячейки с кодом и выводами.
- В каждой задаче приведите код, выведите на экран требуемые значения и добавьте краткий вывод (1-2 фразы) в текстовой ячейке.
- Там, где используется случайная генерация, фиксируйте np.random.seed для воспроизводимости.
- Разрешённые библиотеки: numpy, scipy, statsmodels, matplotlib.

Формат сдачи: файл .ipynb с заполненными ячейками. Перед отправкой перезапустите ноутбук от начала до конца, чтобы **вывод от кода был отображён**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

### Задание 1. Сопоставление статистик

Для каждой ситуации укажите наиболее подходящий критерий из списка: z-тест, t-тест (объединённый), Welch t-тест, F-тест (сравнение дисперсий), критерий хи-квадрат (согласия), однофакторный ANOVA, Tukey HSD, критерий Левена.

**Ситуации:**
1. Нужно проверить, отличается ли средний объём дозы от заданного значения; дисперсия процесса известна.
2. Сравнить средние двух независимых групп при неизвестных, но равных дисперсиях.
3. Сравнить средние двух независимых групп при неизвестных и неравных дисперсиях.
4. Оценить, одинаковы ли средние в трёх и более группах.
5. Проверить, согласуются ли наблюдаемые частоты с распределением Пуассона.

**Что сделать:** для каждой ситуации кратко (1-2 предложения) обоснуйте выбор критерия в MarkDown-ячейке ниже.

**Вывод:** <ВАШ ОТВЕТ>

1. z-тест - Среднее сравниваем с заданным значением, дисперсия генеральной совокупности известна.
2. Объединённый t-тест - Сравниваются средние двух независимых групп с неизвестными, но равными дисперсиями.
3. Welch t-тест - Сравниваются средние двух независимых групп с разными неизвестными дисперсиями.
4. Однофакторный ANOVA - Позволяет проверить равенство средних сразу в трёх и более группах.
5. Критерий хи-квадрат согласия - Сравнивает наблюдаемые частоты с ожидаемыми частотами по распределению Пуассона.

### Задание 2. Тест для дозатора

Дозатор настроен на среднюю дозу 3.00 г. Получено $n = 50$ измерений со средним $\bar X = 3.005$ и известным стандартным отклонением процесса $\sigma = 0.015$.

**Что сделать:**
- проверьте гипотезу $H_0: \mu = 3.00$ при $\alpha = 0.05$;
- постройте 95% доверительный интервал для $\mu$;
- сделайте практический вывод в MarkDown-ячейке: нужно ли перенастраивать дозатор.

**Выведите на экран:** наблюдаемое значение статистики, p-value, границы доверительного интервала.

In [ ]:
mu0, xbar, sigma, n = 3.00, 3.005, 0.015, 50

z = (xbar - mu0) / (sigma / np.sqrt(n))
p = 2 * stats.norm.sf(abs(z))

margin = stats.norm.ppf(0.975) * sigma / np.sqrt(n)
ci_low = xbar - margin
ci_high = xbar + margin

print(f'z-статистика: {z:.4f}')
print(f'p-value: {p:.6f}')
print(f'95% доверительный интервал: [{ci_low:.6f}; {ci_high:.6f}]')

**Вывод:** p-value = 0.0184 < 0.05, поэтому гипотезу о средней дозе 3.00 г отвергаем; 95% доверительный интервал не содержит 3.00 г. Средняя доза немного завышена, поэтому дозатор стоит проверить и при необходимости перенастроить с учётом допустимой погрешности.

### Задание 3. Сравнение прочности материалов

Даны две независимые выборки прочности материалов (в MPa):

A (n=15): 51.99, 49.45, 52.59, 56.09, 49.06, 49.06, 56.32, 53.07, 48.12, 52.17, 48.15, 48.14, 50.97, 42.35, 43.10

B (n=10): 48.63, 45.92, 53.89, 46.55, 43.53, 60.79, 50.65, 52.41, 43.45, 48.73

**Что сделать:**
- проверьте равенство дисперсий (критерий Левена и классический F-тест);
- в зависимости от результата примените объединённый t-тест или Welch t-тест для проверки $H_0: \mu_A = \mu_B$ при $\alpha = 0.05$;
- постройте 95% доверительный интервал для разности средних и интерпретируйте его.

Выведите на экран: p-value критерия Левена, F-статистику и p-value F-теста, статистику и p-value выбранного t-теста, разность средних и границы доверительного интервала.

In [ ]:
A = np.array([51.99,49.45,52.59,56.09,49.06,49.06,56.32,53.07,48.12,52.17,48.15,48.14,50.97,42.35,43.10])
B = np.array([48.63,45.92,53.89,46.55,43.53,60.79,50.65,52.41,43.45,48.73])

p_levene = stats.levene(A, B).pvalue

f = A.var(ddof=1) / B.var(ddof=1)
p_f = 2 * min(stats.f.cdf(f, len(A)-1, len(B)-1),
              stats.f.sf(f, len(A)-1, len(B)-1))

# Выбираем t-тест по результатам проверки дисперсий
equal_var = p_levene > 0.05 and p_f > 0.05
t, p_t = stats.ttest_ind(A, B, equal_var=equal_var)

mean_diff = A.mean() - B.mean()
comparison = CompareMeans(DescrStatsW(A), DescrStatsW(B))
ci = comparison.tconfint_diff(usevar='pooled' if equal_var else 'unequal')

print(f'Левен, p-value: {p_levene:.4f}')
print(f'F-статистика: {f:.4f}, p-value: {p_f:.4f}')
print('Выбранный тест:', 'объединённый t-тест' if equal_var else 'Welch t-тест')
print(f't-статистика: {t:.4f}, p-value: {p_t:.4f}')
print(f'Разность средних A - B: {mean_diff:.4f} MPa')
print(f'95% доверительный интервал: [{ci[0]:.4f}; {ci[1]:.4f}] MPa')

**Вывод:** Оба теста не выявили различий дисперсий (Левен: p = 0.5009, F-тест: p = 0.3271), поэтому использован объединённый t-тест. Разница средних 0.587 MPa статистически незначима (p = 0.7541), а 95% интервал [-3.24; 4.42] MPa включает ноль.

### Задание 4. До и после: уровень глюкозы

Клиническое исследование: у 10 пациентов измерен уровень глюкозы (ммоль/л) до и после приёма нового лекарства.

До: 7.1, 6.8, 7.5, 6.9, 7.3, 6.5, 7.0, 6.7, 7.2, 6.9

После: 6.6, 6.4, 7.1, 6.5, 7.2, 6.3, 6.6, 6.4, 7.0, 6.4

**Что сделать:**
- проверьте нормальность распределения разностей (критерий Шапиро-Уилка);
- в зависимости от результатов проверки на нормальность примените либо парный t-тест при $\alpha = 0.01$, либо критерий Уилкоксона (**только один**, необходимо выбрать и обосновать решение);
- сделайте практический вывод: достоверно ли снижается уровень глюкозы.

**Выведите на экран:** p-value критерия Шапиро-Уилка, статистику и p-value выбранного теста, среднюю разность.

In [ ]:
before = np.array([7.1,6.8,7.5,6.9,7.3,6.5,7.0,6.7,7.2,6.9])
after  = np.array([6.6,6.4,7.1,6.5,7.2,6.3,6.6,6.4,7.0,6.4])
diff = after - before

p_shapiro = stats.shapiro(diff).pvalue

# Разности близки к нормальным, поэтому парный t-тест
# H1: средний уровень после приёма ниже, чем до
result = stats.ttest_rel(after, before, alternative='less')

print(f'Шапиро-Уилк, p-value: {p_shapiro:.6f}')
print(f'Парный t-тест, статистика: {result.statistic:.4f}')
print(f'p-value: {result.pvalue:.8f}')
print(f'Средняя разность после - до: {diff.mean():.4f} ммоль/л')

**Вывод:** Нормальность разностей не отвергается (Шапиро–Уилк: p = 0.1977), поэтому применён односторонний парный t-тест. Уровень глюкозы после приёма статистически значимо ниже при α = 0.01 (p ≈ 0.000011), среднее снижение составляет 0.34 ммоль/л.

### Задание 5. ANOVA на реальных данных

Данные эксперимента по влиянию четырёх типов корма на массу животных (в граммах), по 6 наблюдений в группе:

A: 251, 262, 248, 255, 260, 257
B: 263, 270, 265, 272, 260, 266
C: 268, 275, 271, 280, 277, 269
D: 282, 290, 285, 295, 288, 293

**Что сделать:**
- постройте визуализацию (ящики с усами и точки);
- проверьте нормальность по группам (Шапиро-Уилк) и равенство дисперсий (критерий Левена);
- проведите однофакторный ANOVA;
- при значимом результате выполните попарные сравнения Tukey HSD и определите наиболее эффективный корм.

**Выведите на экран:** p-value Шапиро-Уилка по каждой группе, p-value критерия Левена, F-статистику и p-value ANOVA, таблицу Tukey HSD.

In [ ]:
A = [251,262,248,255,260,257]
B = [263,270,265,272,260,266]
C = [268,275,271,280,277,269]
D = [282,290,285,295,288,293]

groups = [A, B, C, D]
labels = ['A', 'B', 'C', 'D']

plt.figure(figsize=(7, 4))
plt.boxplot(groups)
for i, group in enumerate(groups, start=1):
    plt.scatter(np.linspace(i-0.08, i+0.08, len(group)), group)
plt.xticks([1, 2, 3, 4], labels)
plt.xlabel('Тип корма')
plt.ylabel('Масса, г')
plt.title('Масса животных по типам корма')
plt.show()

for label, group in zip(labels, groups):
    print(f'Шапиро-Уилк, группа {label}, p-value: {stats.shapiro(group).pvalue:.4f}')

p_levene = stats.levene(A, B, C, D).pvalue
f, p_anova = stats.f_oneway(A, B, C, D)
print(f'Левен, p-value: {p_levene:.4f}')
print(f'ANOVA, F-статистика: {f:.4f}, p-value: {p_anova:.10f}')

if p_anova < 0.05:
    values = A + B + C + D
    names = ['A']*6 + ['B']*6 + ['C']*6 + ['D']*6
    tukey = pairwise_tukeyhsd(values, names, alpha=0.05)
    print(tukey)

for label, group in zip(labels, groups):
    print(f'Средняя масса, корм {label}: {np.mean(group):.2f} г')

**Вывод:** Тесты Шапиро–Уилка и Левена не выявили нарушений предпосылок ANOVA; средние массы различаются (p ≈ 1.8·10⁻⁹). Корм D даёт наибольшую среднюю массу (288.83 г) и по Tukey HSD значимо превосходит каждый из остальных кормов.

### Задание 6. A/B-тест по конверсии

Проводится A/B-тест на лендинге. В контрольной группе из 1000 пользователей конверсий 85, в тестовой группе из 1000 пользователей конверсий 120.

**Что сделать:**
- постройте 95% доверительные интервалы Уилсона для конверсии каждой группы;
- проверьте, значимо ли различаются конверсии (критерий хи-квадрат на таблице 2x2 или z-тест для двух долей) при $\alpha = 0.05$;
- постройте бутстрэп-интервал для разности конверсий (тест минус контроль);
- сделайте вывод, стоит ли выкатывать тестовый вариант.

**Выведите на экран:** границы интервалов Уилсона по группам, статистику и p-value выбранного теста, границы бутстрэп-интервала для разности и точечную разность конверсий.

In [ ]:
conv_c, n_c = 85, 1000     # контроль
conv_v, n_v = 120, 1000    # тест

p_c = conv_c / n_c
p_v = conv_v / n_v
ci_c = proportion_confint(conv_c, n_c, alpha=0.05, method='wilson')
ci_v = proportion_confint(conv_v, n_v, alpha=0.05, method='wilson')
z, p = proportions_ztest([conv_v, conv_c], [n_v, n_c])

# Бутстрэп разности конверсий
np.random.seed(42)
control = np.array([1]*conv_c + [0]*(n_c-conv_c))
variant = np.array([1]*conv_v + [0]*(n_v-conv_v))
boot_diff = []
for _ in range(10000):
    sample_c = np.random.choice(control, n_c, replace=True)
    sample_v = np.random.choice(variant, n_v, replace=True)
    boot_diff.append(sample_v.mean() - sample_c.mean())

boot_ci = np.percentile(boot_diff, [2.5, 97.5])

print(f'Контроль, конверсия: {p_c:.4f}, интервал Уилсона: [{ci_c[0]:.4f}; {ci_c[1]:.4f}]')
print(f'Тест, конверсия: {p_v:.4f}, интервал Уилсона: [{ci_v[0]:.4f}; {ci_v[1]:.4f}]')
print(f'z-статистика: {z:.4f}, p-value: {p:.6f}')
print(f'Разность конверсий: {p_v-p_c:.4f} ({(p_v-p_c)*100:.1f} п.п.)')
print(f'95% бутстрэп-интервал разности: [{boot_ci[0]:.4f}; {boot_ci[1]:.4f}]')

**Вывод:** Конверсия выросла с 8.5% до 12.0% (+3.5 п.п.), различие значимо (p = 0.0099), а 95% бутстрэп-интервал [0.008; 0.062] не содержит ноль. Тестовый вариант можно рекомендовать к выкатке, если остальные важные метрики не ухудшились.